In [1]:
'''
this is the main script where the matching of the roll numbers and the summoning of the feed will be taking place
script 3.2
~ssmjtc
'''

'\nthis is the main script where the matching of the roll numbers and the summoning of the feed will be taking place\nscript 3.2\n~ssmjtc\n'

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install InsightFace
!pip install insightface onnxruntime

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.1 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp311-cp311-linux_x86_64.whl size=1060429 sha256=a1bcc31a003776564e835980698cb7cec45c7083efc85fdf78d9d2a2e0aae401
  Stored in directory: /root/.cache/pip/wheels/27/d8/22/f52d858d16cd06e7b2e6aad34a1777dcfaf000be833bbf8146
Successfully built insightface


**dependencies**

In [16]:
import cv2
import numpy as np
import pickle
import threading
import queue
import time
import os
from sklearn.metrics.pairwise import cosine_similarity
from insightface.app import FaceAnalysis
from google.colab.patches import cv2_imshow

from IPython.display import display, clear_output, Image
import random

print("SUCCESS")

SUCCESS


**FETCHING EMBEDD**

In [4]:
#embedding structure
import pickle

with open("/content/drive/MyDrive/Colab Notebooks/RP/embeddings.pkl", "rb") as f:
    data = pickle.load(f)

print(type(data))
print(len(data))
print(data.keys())  # or list(data.keys()) if it's a dict

<class 'dict'>
2
dict_keys(['embeddings', 'metadata'])


**recrded**

In [22]:
class LiveFaceMatcherMulti:
    def __init__(self, embeddings_path, provider='CPUExecutionProvider'):
        self.app = FaceAnalysis(providers=[provider])
        self.app.prepare(ctx_id=0, det_size=(480, 480))
        self.data = self.load_embeddings(embeddings_path)

    def load_embeddings(self, path):
        with open(path, "rb") as f:
            return pickle.load(f)

    def get_embeddings_for_id(self, target_id):
        embeddings = self.data["embeddings"]
        metadata = self.data["metadata"]
        return np.array([emb for emb, meta in zip(embeddings, metadata) if str(meta["id"]) == str(target_id)])

    def is_match(self, query_emb, target_embs, threshold=0.5):
        if len(target_embs) == 0:
            return False
        sims = cosine_similarity([query_emb], target_embs)[0]
        return sims.max() > (1 - threshold)

    def process_video_live_multi(self, video_path, target_ids, output_path="processed_output.mp4"):
        # Prepare embeddings dict for all target IDs
        target_embs_dict = {tid: self.get_embeddings_for_id(tid) for tid in target_ids}
        status = {tid: False for tid in target_ids}  # Track presence

        # Assign random colors to each ID
        colors = {tid: tuple(np.random.randint(0, 256, 3).tolist()) for tid in target_ids}

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Failed to open video file: {video_path}")
            return

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        print(f"Saving processed video to: {os.path.abspath(output_path)}")

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            faces = self.app.get(frame)
            for face in faces:
                emb = face.embedding
                for tid, target_embs in target_embs_dict.items():
                    if self.is_match(emb, target_embs):
                        bbox = face.bbox.astype(int)
                        color = colors[tid]
                        cv2.rectangle(frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                        cv2.putText(frame, f"ID: {tid}", (bbox[0], bbox[1] - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                        status[tid] = True

            # Draw status bar at top-left
            y0, dy = 30, 25
            for i, tid in enumerate(target_ids):
                text = f"ID {tid}: {'Present' if status[tid] else 'Absent'}"
                color = colors[tid] if status[tid] else (0, 0, 255)
                cv2.putText(frame, text, (10, y0 + i*dy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            # Write frame to output
            out.write(frame)

            # Optional live display in Colab
            _, buffer = cv2.imencode(".png", frame)
            clear_output(wait=True)
            display(Image(data=buffer.tobytes()))
            time.sleep(1 / max(fps, 30))

        cap.release()
        out.release()
        clear_output(wait=True)
        print("Processing complete. Video saved.")

        # Print final status for all IDs
        print("\n--- Detection Status ---")
        for tid, present in status.items():
            print(f"ID {tid}: {'Present' if present else 'Absent'}")


# ---------------- Main ---------------- #
video_path = "/content/drive/MyDrive/Colab Notebooks/RP/test/test1_clear.mp4"
embeddings_path = "/content/drive/MyDrive/Colab Notebooks/RP/embeddings.pkl"

# Ask user to input multiple roll numbers
roll_input = input("Enter roll IDs to locate (comma-separated, e.g., 1,2,3): ")
target_ids = [int(r.strip()) for r in roll_input.split(",") if r.strip().isdigit()]

matcher = LiveFaceMatcherMulti(embeddings_path)
matcher.process_video_live_multi(video_path, target_ids, output_path="processed_output.mp4")


Processing complete. Video saved.

--- Detection Status ---
ID 1: Present
ID 2: Present
ID 6: Absent
